# GPT-OSS 20B LoRA fine-tuning (current pipeline)
This notebook mirrors your current flow: install, login, load dataset, tokenize, load model, quick test, LoRA, train, save/push, and inference.

In [12]:
!pip install torch transformers trl peft datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 18.1 MB/s eta 0:00:00


In [13]:
import torch, transformers,trl, peft, datasets, huggingface_hub

In [ ]:
print(torch.__version__, "\n", transformers.__version__, "\n", trl.__version__, "\n", peft.__version__, "\n", datasets.__version__, "\n", huggingface_hub.__version__)

2.9.0+cu126 
 4.57.3 
 0.26.2 
 0.18.0 
 4.0.0 
 0.36.0


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
from datasets import load_dataset
dataset=load_dataset("HuggingFaceH4/Multilingual-Thinking",split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
dataset[0]

{'reasoning_language': 'French',
 'developer': 'You are an AI chatbot with a lively and energetic personality.',
 'user': 'Can you show me the latest trends on Twitter right now?',
 'analysis': "D'accord, l'utilisateur demande les tendances Twitter les plus récentes. Tout d'abord, je dois vérifier si j'ai accès à des données en temps réel. Étant donné que je ne peux pas naviguer sur Internet ou accéder directement à l'API de Twitter, je ne peux pas fournir des tendances en direct. Cependant, je peux donner quelques conseils généraux sur la façon de les trouver.\n\nJe devrais préciser que les tendances Twitter évoluent rapidement et sont spécifiques à chaque région. Je pourrais suggérer de consulter la section «\xa0En vogue\xa0» sur l'application ou le site web. Aussi, l'utilisation de hashtags et le suivi d'utilisateurs pertinents pourraient être utiles. Il est important de souligner que les tendances varient selon la région et l'heure de la journée. Je devrais garder un ton amical et 

In [6]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("openai/gpt-oss-20b")
messages=dataset[0]['messages']
text=tokenizer.apply_chat_template(messages, tokenize=False)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [51]:
print(text)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-01-08

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are an AI chatbot with a lively and energetic personality.

<|end|><|start|>user<|message|>Can you show me the latest trends on Twitter right now?<|end|><|start|>assistant<|channel|>analysis<|message|>D'accord, l'utilisateur demande les tendances Twitter les plus récentes. Tout d'abord, je dois vérifier si j'ai accès à des données en temps réel. Étant donné que je ne peux pas naviguer sur Internet ou accéder directement à l'API de Twitter, je ne peux pas fournir des tendances en direct. Cependant, je peux donner quelques conseils généraux sur la façon de les trouver.

Je devrais préciser que les tendances Twitter évoluent rapidement et sont spécifiques à chaque 

In [7]:
inputs=tokenizer(text)
print(inputs)

{'input_ids': [200006, 17360, 200008, 3575, 553, 17554, 162016, 11, 261, 4410, 6439, 2359, 22203, 656, 7788, 17527, 558, 87447, 100594, 25, 220, 1323, 19, 12, 3218, 198, 6576, 3521, 25, 220, 1323, 21, 12, 2290, 12, 3062, 279, 30377, 289, 25, 14093, 279, 2, 13888, 18403, 25, 8450, 11, 49159, 11, 1721, 13, 21030, 2804, 413, 7360, 395, 1753, 3176, 13, 200007, 200006, 77944, 200008, 2, 68406, 279, 42743, 289, 6439, 25, 12911, 279, 3575, 553, 448, 20837, 165942, 483, 261, 56722, 326, 55114, 24476, 364, 200007, 200006, 1428, 200008, 8475, 481, 2356, 668, 290, 6898, 20464, 402, 10462, 1849, 1954, 30, 200007, 200006, 173781, 200005, 35644, 200008, 35, 95247, 11, 305, 140803, 26737, 1341, 152797, 10462, 1341, 2932, 3146, 109284, 13, 45412, 272, 89989, 11, 1264, 18435, 93671, 1784, 441, 12480, 65948, 1221, 731, 30580, 469, 11160, 58265, 13, 10055, 26712, 52079, 661, 1264, 453, 43329, 2173, 34983, 4615, 1512, 7380, 2031, 120555, 52317, 1221, 305, 6, 8180, 334, 10462, 11, 1264, 453, 43329, 2173, 8

In [8]:
from transformers import AutoModelForCausalLM
checkpoint="openai/gpt-oss-20b"
model=AutoModelForCausalLM.from_pretrained(
    checkpoint, 
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

In [9]:
inputs_ids=tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

output_ids = model.generate(inputs_ids, max_new_tokens=256)
tokenizer.batch_decode(output_ids,skip_special_tokens=True)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['systemYou are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-01-08\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.developer# Instructions\n\nreasoning language: French\n\nYou are an AI chatbot with a lively and energetic personality.\n\nuserCan you show me the latest trends on Twitter right now?assistantfinalHey there!  While I can\'t check Twitter (X) in real-time or access live data, I can share some tips to help you spot the latest trends:\n\n1. **Open the "Trending" tab** on the Twitter app or website – it updates constantly!  \n2. **Search for hashtags** like #Trending or #Viral to see what’s blowing up.  \n3. **Follow accounts** that curate trends (e.g., @TrendingNow, @ViralThreads).  \n4. **Check regional trends** – they often differ by location!  \n\nRemember, trends are *super fast-moving* and often tied to pop culture, memes, or breaking news. For example, r

In [85]:
raw_messages=[
    {"role":"system",'content': 'reasoning language: Telugu \n\nYou are an AI chatbot with a lively and energetic personality.'},
    {'role':'user', 'content': 'what is the national animal of India.'},
]

inputs_ids=tokenizer.apply_chat_template(
    raw_messages,
    add_generation_prompt=False,
    return_tensors="pt",
    # tokenize =False,
).to(model.device)

# inputs_ids
output_ids = model.generate(inputs_ids, max_new_tokens=1024)


In [86]:
response = tokenizer.batch_decode(output_ids,skip_special_tokens=False)[0]
print(response)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-01-08

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

reasoning language: Telugu 

You are an AI chatbot with a lively and energetic personality.

<|end|><|start|>user<|message|>what is the national animal of India.<|end|><|start|>assistant<|channel|>analysis<|message|>సరే, వినియోగదారు అడుగుతున్నది భారతదేశపు జాతీయ జంతువు ఏమిటో. భారతదేశపు నేచురల్ సెక్యురిటీ గురించి తెలుసుకోవాలి. భారతదేశపు అధికారిక జాతీయ జంతువు "సింహం," అని గుర్తు. ఇది పౌరాణికమైన ప్రాధాన్యత కలిగి ఉంది, హిందూ మతం, చరిత్రలో గట్టి స్థానం కలిగి ఉంటుంది. సింహాన్ని సాధారణంగా శక్తి, ధైర్యం వంటి గుణాలతో అనుసంధానించబడుతుంది, మరియు భారతీయ మసాజ్‌లలో, చారిత్రాత్మకంగా ముఖ్యమైన భాగంగా ఉంటుంది. అదనంగా, పత్రంలో ఎక్కడో "కోటి జంతువు" కూడా ఉందని గుర్తుంచుకుని, రెండు ముఖ్యమైన జంతువులు గురించి మాట్లాడా

In [ ]:
from peft import LoraConfig, get_peft_model
peft_config=LoraConfig(
    r=8,
    lora_alpha=16, 
    target_modules = 'all-linear',
)

peft_model=get_peft_model(model,peft_config)
peft_model.print_trainable_parameters()

trainable params: 3,981,312 || all params: 20,918,738,496 || trainable%: 0.0190


In [10]:
from trl import SFTConfig, SFTTrainer
training_args=SFTConfig(
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    max_length=1024,
    output_dir="./gpt-oss-ft",
)
trainer=SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
    )
trainer.train()

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998}.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


Step,Training Loss
10,1.706500
20,1.276400
30,1.155700
40,1.130800
50,1.099500
60,1.116500


wandb: WARNING URL not available in offline run


TrainOutput(global_step=63, training_loss=1.2399269474877253, metrics={'train_runtime': 6989.8169, 'train_samples_per_second': 0.143, 'train_steps_per_second': 0.009, 'total_flos': 1.0161438882400973e+17, 'train_loss': 1.2399269474877253, 'entropy': 1.021517388522625, 'num_tokens': 832648.0, 'mean_token_accuracy': 0.708193090558052, 'epoch': 1.0})

In [15]:
trainer.save_model(training_args.output_dir)

In [16]:
trainer.push_to_hub("ramyeturi/gpt-oss-20b-finetuned-multilingual-thinking")

wandb: WARNING URL not available in offline run


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gpt-oss-ft/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

  ...adapter_model.safetensors:   4%|3         |  585kB / 16.0MB            

  ...35366.e3f3ee8ea024.3651.0:   4%|3         |   329B / 9.30kB            

  ...-oss-ft/training_args.bin:   4%|3         |   220B / 6.22kB            

CommitInfo(commit_url='https://huggingface.co/ramyeturi/gpt-oss-ft/commit/f453422823fea538c28dc7895ce3da858440a698', commit_message='ramyeturi/gpt-oss-20b-finetuned-multilingual-thinking', commit_description='', oid='f453422823fea538c28dc7895ce3da858440a698', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ramyeturi/gpt-oss-ft', endpoint='https://huggingface.co', repo_type='model', repo_id='ramyeturi/gpt-oss-ft'), pr_revision=None, pr_num=None)

In [18]:
from peft import PeftModel
ft_model=PeftModel.from_pretrained(
    model,
    "ramyeturi/gpt-oss-ft"
)

ft_model=ft_model.merge_and_unload()

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

In [77]:
raw_messages=[
    {"role":"system",'content': 'reasoning language: Hindi \n\nYou are an AI chatbot with a lively and energetic personality.'},
    {'role':'user', 'content': 'what is the national animal of India.'},
]

inputs_ids=tokenizer.apply_chat_template(
    raw_messages,
    add_generation_prompt=False,
    return_tensors="pt"
).to(ft_model.device)

output_ids = ft_model.generate(inputs_ids, max_new_tokens=1024)


In [78]:
response = tokenizer.batch_decode(output_ids,skip_special_tokens=False)[0]
print(response.format())

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-01-08

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

reasoning language: Hindi 

You are an AI chatbot with a lively and energetic personality.

<|end|><|start|>user<|message|>what is the national animal of India.<|end|><|start|>assistant<|channel|>analysis<|message|>भले ही इस प्रश्न का उत्तर सामान्य तथ्य हो सकता है, मैं इसका उत्तर दिलचस्प तरीके से देना चाहूंगा। मैं राष्ट्रीय पशु के इतिहास और महत्व पर गौर करता हूं। यह भारत में अपनी महत्ता के लिए मैन्टेलिंग भालू क्यों चुना गया? मैं इसके सांस्कृतिक और ऐतिहासिक संदर्भों को जोड़ने का प्रयास करूं। यह भारत की विशिष्ट संस्कृतियों के साथ इसका तालमेल क्यों रहा है? क्या कोई रोचक ऐतिहासिक घटनाएं या धार्मिक विश्वास इसके चयन को प्रभावित करते हैं? जवाब को संक्षिप्त, फिर भी सूचनात्मक रखना मुझे प्रेरित करता है।